## Setting up Configuration

In [0]:
%run ./00_setup_config

## Reading data from weather_curated table

In [0]:
df_curated = spark.table("internship_databricks_ws.default.weather_curated")

## Splitting weather_curated + adding unique id

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max


df_dims_city = spark.table("internship_databricks_ws.default.weather_dims_city").select("city", "city_id")
df_cities = df_curated.select("city").distinct()

new_cities = df_cities.join(df_dims_city, on="city", how="left_anti")
max_id = df_dims_city.agg(spark_max("city_id")).collect()[0][0] or 0
if new_cities.count() > 0:
    new_cities = new_cities.withColumn("city_id", row_number().over(Window.orderBy("city")) + max_id).select("city_id", "city")
    new_cities.write.format("delta").mode("append").saveAsTable("internship_databricks_ws.default.weather_dims_city")
df_dims_city = spark.table("internship_databricks_ws.default.weather_dims_city")
df_dims_city.show(truncate=False)

## Joining weather_curated with df_dims_city by matching cities

In [0]:
df_fact_new = df_curated.join(df_dims_city.select("city", "city_id"), on="city", how="left") \
    .select(
        "city_id",
        "weather_time_pkt",
        "temperature_c",
        "humidity_pct",
        "feels_like_c",
        "precipitation_mm",
        "weather_code",
        "pressure_hpa",
        "windspeed_kmh",
        "winddirection_deg",
        "cloud_cover_pct",
        "is_day",
        "loading_time"
    )

try:
    df_fact_existing = spark.table("internship_databricks_ws.default.weather_fact_weather")
    df_fact_to_append = df_fact_new.join(
        df_fact_existing.select("city_id","weather_time_pkt"),
        on=["city_id","weather_time_pkt"], how="left_anti"
    )
except Exception:
    df_fact_to_append = df_fact_new

df_fact_to_append.show(truncate=False)

## Creating weather_dims_city table + writing df_dims_city's data in it

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dims_city
USING DELTA
LOCATION '{gold_path}weather_dims_city/'
""")

print("Written to gold layer")

In [0]:
%sql
Select * from weather_dims_city

## Creating weather_fact_weather table + writing df_fact_weather's data in it

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_fact_weather
USING DELTA
LOCATION '{gold_path}weather_fact_weather/'
""")
df_fact_to_append.write.format("delta").mode("append").saveAsTable("internship_databricks_ws.default.weather_fact_weather")
print("Appended new readings to fact_weather")

## Aggregating weather_summary for cities

In [0]:
%sql
CREATE OR REPLACE TABLE internship_databricks_ws.default.weather_summary
USING DELTA
LOCATION 'abfss://gold@internshipdlsa01.dfs.core.windows.net/weather_summary/'
AS
SELECT
    d.city,
    ROUND(AVG(f.temperature_c), 1) AS avg_temp_c,
    ROUND(MAX(f.temperature_c), 1) AS max_temp_c,
    ROUND(MIN(f.temperature_c), 1) AS min_temp_c,
    ROUND(AVG(f.humidity_pct), 1) AS avg_humidity_pct,
    ROUND(AVG(f.windspeed_kmh), 1) AS avg_windspeed_kmh,
    COUNT(*) AS reading_count
FROM internship_databricks_ws.default.weather_fact_weather f
JOIN internship_databricks_ws.default.weather_dims_city d ON f.city_id = d.city_id
GROUP BY d.city
ORDER BY d.city

## Finding rainy cities

In [0]:
%sql
CREATE OR REPLACE TABLE internship_databricks_ws.default.weather_rainy_cities
USING DELTA
LOCATION 'abfss://gold@internshipdlsa01.dfs.core.windows.net/weather_rainy_cities/'
AS
SELECT c.city, f.precipitation_mm, f.humidity_pct
FROM internship_databricks_ws.default.weather_dims_city c join 
weather_fact_weather f on c.city_id = f.city_id
WHERE precipitation_mm > 0

## Calculating temperature range for cities

In [0]:
%sql
Create or replace table internship_databricks_ws.default.weather_cities_range
using delta location
'abfss://gold@internshipdlsa01.dfs.core.windows.net/weather_cities_range/'
as
Select c.city, avg(f.temperature_c) as avg_temperature_c, ROUND(max(f.temperature_c) - min(f.temperature_c),2) as temp_range from internship_databricks_ws.default.weather_dims_city c join weather_fact_weather f on c.city_id = f.city_id
group by c.city
order by temp_range